# chesszero — self-play et boucle AlphaZero sur Kaggle (étapes 6e et 7e)

**Quatre modes**, choisis dans la cellule 0 (`MODE`) :
- `"check"` (**défaut, sans GPU**) : vérifie le code (`COMMIT`), les entrées trouvées dans `/kaggle/input` (réseau de
  départ, données Lichess, sortie précédente exactement une, `EXPECTED_SESSION`) et ce que la session ferait.
  **Rien n'est écrit dans `/kaggle/working`.** À lancer avant chaque session de boucle (session interactive, sans GPU,
  quelques minutes, pas de quota GPU).
- `"loop"` : **session de la boucle AlphaZero** (`scripts/alphazero_session.py`) pendant `LOOP_MINUTES` : self-play,
  tours d'entraînement, arènes, verdicts, rapport `loop/reports/session_XXXX.md` (« Actions pour toi » en tête).
  Première session : 220 min ; sessions suivantes : 660 min. **GPU requis.**
- `"bench"` : essai de débit du self-play (15 min, rien dans le tampon).
- `"full"` : session de self-play seul (`FULL_MINUTES`), ancien mode de l'étape 6.

**Reproductibilité** : `COMMIT` est le hash complet (40 caractères) du code utilisé ; la cellule 1 clone le dépôt, se place
sur ce commit et refuse tout autre état. Chaque session ajoute ce hash à son rapport. Ne changer `COMMIT` que volontairement,
après validation.

**Réglages de la session** (panneau de droite) : *Accelerator* = GPU T4 x2 (un seul est utilisé) en `loop` / `bench` /
`full` ; *Internet* = **On** ; *Persistence* = No persistence.

**Entrées** (*Add Input*) — exactement ceci, pas plus :
1. Dataset `chesszero-kaggle-sup` (le `best.pt` de l'étape 4b, SHA-256 `c1f616c6…18b33` vérifié) ;
2. Dataset `lichess-24m` ;
3. **la sortie de la session précédente** (*Your Work → ce notebook*) : une seule. Première session de boucle = la sortie
   de la session de self-play seul. Ensuite = la sortie de la dernière session de boucle terminée.

**`EXPECTED_SESSION`** = numéro de la dernière session de boucle contenue dans l'entrée (**0** pour la sortie de
self-play seul). Le rapport de chaque session donne la valeur suivante en deuxième ligne.

**Sorties** (`/kaggle/working`) : `selfplay/` (tampon), `sp_train/` (entraînement, état), `loop/` (état, arènes,
rapports, `history.csv`, `progress.png`). Le dépôt et les tables Syzygy sont dans `/tmp` : ils ne gonflent pas la sortie.

In [ ]:
# 0. Réglages ---------------------------------------------------------------------------------------------
MODE = "check"            # "check" (défaut, sans GPU, rien d'écrit) | "loop" | "bench" | "full"
COMMIT = "a1296f47b308e694b67bdc81d9fd2d8c28a31084"     # hash COMPLET du code utilisé (reproductibilité) : à changer volontairement, jamais None
LOOP_MINUTES = 660        # loop : budget de la session. 1re session : 220 ; ensuite 660 (1 h de marge sous les 12 h)
EXPECTED_SESSION = 1      # loop / check : dernière session de boucle contenue dans l'entrée (0 = sortie de self-play seul)
RESET_LINE = None         # loop / check : None, ou une note ("politique aplatie, session 1") = ligne d'entraînement remise sur
                          # le réseau accepté au début de la session (poids, AdamW neuf, compteurs de rejets ; tampon gardé)
BENCH_MINUTES = 15        # bench : durée totale (5 min de débit + 10 min de parties)
FULL_MINUTES = 660        # full : budget de la session de self-play seul
SELFPLAY_ONLY = True      # full : True = self-play seul ; False = + tours d'entraînement (provisoire, remplacé par loop)
WORKERS = None            # processus de self-play (None = cœurs CPU - 1, au plus 3)
GAMES_PER_WORKER = None   # parties menées ensemble par processus (None = [selfplay] de configs/kaggle.toml : 64)
EXPECTED_SHA256 = "c1f616c6eb908a89645b611e61229828a900c09d40c3477f8ba0284b22418b33"  # best.pt de kaggle_sup (pas 92 000)

REPO_URL = "https://github.com/Enz091/Project_chess.git"

import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
REPO_DIR = Path("/tmp/Project_chess")   # hors de /kaggle/working : pas dans la sortie
SYZYGY = Path("/tmp/syzygy")            # tables WDL (396 Mo), retéléchargées à chaque session
assert MODE in ("check", "loop", "bench", "full"), MODE
if MODE == "loop":
    assert 30 <= LOOP_MINUTES <= 660, "LOOP_MINUTES entre 30 et 660 (coupure de Kaggle à 12 h)"
if not re.fullmatch(r"[0-9a-f]{40}", str(COMMIT)):
    raise ValueError("COMMIT doit être un hash complet de 40 caractères hexadécimaux (pas None, pas « main ») : chaque "
                     "session doit être reproductible. Voir le journal (instructions de la première session).")


def run(cmd, cwd=None):
    # Lance une commande et affiche sa sortie au fil de l'eau ; lève une erreur si elle échoue.
    proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if code:
        raise RuntimeError(f"échec ({code}) : {' '.join(str(c) for c in cmd)}")
    return code


print(f"MODE = {MODE} ; COMMIT = {COMMIT} ; EXPECTED_SESSION = {EXPECTED_SESSION}")

In [ ]:
# 1. Dépôt au commit figé et installation (sans réinstaller torch) -------------------------------------------------
import importlib
import importlib.metadata as md

torch_before = md.version("torch")
if (REPO_DIR / ".git").exists():
    run(["git", "fetch", "--quiet", "origin"], cwd=REPO_DIR)
else:
    run(["git", "clone", "--quiet", REPO_URL, REPO_DIR])          # dépôt complet : ~8 Mo
run(["git", "checkout", "--quiet", "--detach", COMMIT], cwd=REPO_DIR)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
assert head == COMMIT, f"le dépôt est sur {head}, pas sur COMMIT {COMMIT}"
run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR])  # pas de torch dans les dépendances
sys.path.insert(0, str(REPO_DIR / "src"))  # le noyau ne voit pas un paquet installé après son démarrage
importlib.invalidate_caches()
run(["git", "log", "-1", "--format=commit %H%n%s"], cwd=REPO_DIR)
assert md.version("torch") == torch_before, "pip a modifié torch : à ne pas faire sur Kaggle"
print("torch inchangé :", torch_before)

In [ ]:
# 2. Environnement ------------------------------------------------------------------------------------------------
import platform

import torch

print("Python :", platform.python_version(), "| torch :", torch.__version__, "| CUDA :", torch.version.cuda)
cores = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
WORKERS = WORKERS or max(1, min(3, cores - 1))   # un cœur reste libre (processus principal, mesures)
print(f"Cœurs CPU : {cores} -> {WORKERS} processus de self-play")
if Path("/proc/meminfo").exists():
    mem = {l.split(":")[0]: int(l.split()[1]) for l in Path("/proc/meminfo").read_text().splitlines()[:3]}
    print(f"RAM : {mem['MemTotal'] / 1e6:.0f} Go (libre {mem['MemAvailable'] / 1e6:.0f} Go) ; "
          f"disque /kaggle/working : {shutil.disk_usage(WORKING).free / 1e9:.0f} Go libres")
if MODE == "check":
    print("Mode check : GPU non requis" + (" (aucun GPU vu)" if not torch.cuda.is_available() else ""))
elif not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU : Session options -> Accelerator -> GPU T4 x2")
else:
    p = torch.cuda.get_device_properties(0)
    print(f"GPU : {p.name}, {p.total_memory / 1e9:.0f} Go, {torch.cuda.device_count()} visible(s) (le premier est utilisé)")

In [ ]:
# 3. Entrées : réseau de départ (SHA-256), données Lichess, sortie précédente (exactement une) ---------------------
from chesszero.kaggle_inputs import describe_previous, discover, select_previous

INIT = HUMAN = PREVIOUS = None
PREVIOUS_LIST = []
if MODE in ("check", "loop"):
    found = discover(INPUT_ROOT, EXPECTED_SHA256, need_human=True)
    INIT, HUMAN, PREVIOUS_LIST = found["init"], found["human"], found["previous"]
    problems = list(found["problems"])
    PREVIOUS, more = select_previous(found, EXPECTED_SESSION)
    problems += more
    print("Réseau de départ :", INIT or "INTROUVABLE", "(SHA-256 conforme)" if INIT else "")
    for n in found["notes"]:
        print(n)
    print("Sorties précédentes trouvées :", [describe_previous(p) for p in PREVIOUS_LIST] or "aucune")
    if problems:
        raise RuntimeError("ENTRÉES REFUSÉES :\n- " + "\n- ".join(problems))
    print("Entrée retenue :", PREVIOUS)
else:  # bench / full : comportement de l'étape 6e
    import hashlib

    def sha256(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for block in iter(lambda: f.read(1 << 20), b""):
                h.update(block)
        return h.hexdigest()

    candidates = [p for p in INPUT_ROOT.glob("**/*.pt") if p.stat().st_size > 10_000_000 and "sp_train" not in p.parts]
    INIT = next((p for p in candidates if sha256(p) == EXPECTED_SHA256), None)
    if INIT is None:
        raise FileNotFoundError("Réseau de départ introuvable : ajouter le Dataset contenant best.pt (SHA-256 "
                                f"{EXPECTED_SHA256[:8]}…). Fichiers .pt vus : {[str(p) for p in candidates]}")
    print("Réseau de départ :", INIT, "(SHA-256 conforme)")
    HUMAN = next((p.parent for p in sorted(INPUT_ROOT.glob("**/train_00000.npz"))), None)
    print("Données Lichess :", HUMAN or "absentes")
    if MODE == "full" and not SELFPLAY_ONLY and HUMAN is None:
        raise FileNotFoundError("SELFPLAY_ONLY = False demande le Dataset lichess-24m (Add Input)")
    PREVIOUS_LIST = sorted({p.parent for p in INPUT_ROOT.glob("**/selfplay/*.npz") if p.with_suffix(".json").exists()})
    TRAIN_STATE = next((p.parent for p in sorted(INPUT_ROOT.glob("**/sp_train/train_state.pt"))), None)
    print("Tampon des sessions précédentes :", [str(p) for p in PREVIOUS_LIST] or "aucun (premier remplissage)")
    print("État d'entraînement précédent :", TRAIN_STATE or "aucun")

In [ ]:
# 4. Mode check : vérification complète du démarrage, RIEN n'est écrit dans /kaggle/working ---------------------------
if MODE == "check":
    cmd = [sys.executable, REPO_DIR / "scripts" / "alphazero_session.py", "--check-inputs", "--work", WORKING,
           "--resume-from", PREVIOUS, "--expect-session", EXPECTED_SESSION, "--minutes", LOOP_MINUTES,
           "--init", INIT, "--human", HUMAN, "--syzygy", SYZYGY, "--expect-init-sha256", EXPECTED_SHA256]
    if RESET_LINE:
        cmd += ["--reset-line", RESET_LINE]
    run(cmd)
    leftovers = sorted(p.name for p in WORKING.iterdir()) if WORKING.exists() else []
    print("\n/kaggle/working après la vérification :", leftovers or "vide (conforme : rien n'a été écrit)")
    print("\nTout est conforme ? Passer MODE = \"loop\" (cellule 0), puis Save Version -> Save & Run All (Commit).")

In [ ]:
# 5. Tables Syzygy WDL (3-4-5 pièces, miroir Lichess, chaque fichier vérifié par SHA-256) ----------------------------
if MODE != "check":
    run([sys.executable, REPO_DIR / "scripts" / "download_syzygy.py", "--out", SYZYGY, "--wdl-only"])
else:
    print("Mode check : tables non téléchargées")

In [ ]:
# 6. Session : boucle, essai de débit ou self-play seul -------------------------------------------------------------
if MODE == "loop":
    run([sys.executable, REPO_DIR / "scripts" / "alphazero_session.py", "--work", WORKING, "--resume-from", PREVIOUS,
         "--expect-session", EXPECTED_SESSION, "--minutes", LOOP_MINUTES, "--init", INIT, "--human", HUMAN,
         "--syzygy", SYZYGY, "--workers", WORKERS, "--expect-init-sha256", EXPECTED_SHA256,
         *(["--games-per-worker", GAMES_PER_WORKER] if GAMES_PER_WORKER else []),
         *(["--reset-line", RESET_LINE] if RESET_LINE else [])])
elif MODE in ("bench", "full"):
    cmd = [sys.executable, REPO_DIR / "scripts" / "selfplay_session.py", MODE, "--work", WORKING, "--init", INIT,
           "--syzygy", SYZYGY, "--workers", WORKERS]
    if GAMES_PER_WORKER:
        cmd += ["--games-per-worker", GAMES_PER_WORKER]
    if MODE == "bench":
        cmd += ["--minutes", BENCH_MINUTES]
    else:
        cmd += ["--minutes", FULL_MINUTES]
        if PREVIOUS_LIST:
            cmd += ["--inputs", *PREVIOUS_LIST]
        if not SELFPLAY_ONLY:
            cmd += ["--with-training", "--human", HUMAN]
            if TRAIN_STATE:
                cmd += ["--train-state", TRAIN_STATE]
    run(cmd)
else:
    print("Mode check : aucune session lancée")

In [ ]:
# 7. Contenu de la sortie -----------------------------------------------------------------------------------------
if MODE != "check":
    for d in sorted(p for p in WORKING.iterdir() if p.is_dir()):
        size = sum(f.stat().st_size for f in d.rglob("*") if f.is_file())
        print(f"{d.name:<16} {size / 1e6:8.1f} Mo  {sum(1 for _ in d.rglob('*.npz'))} shards")
    reports = sorted((WORKING / "loop" / "reports").glob("session_*.md"))
    if MODE == "loop" and reports:
        print("\n" + "=" * 100 + f"\nRAPPORT {reports[-1].name} (début)\n" + "=" * 100)
        print("\n".join(reports[-1].read_text(encoding="utf-8").splitlines()[:40]))
    print("\nÀ la session suivante : Add Input -> Your Work -> ce notebook (cette version), retirer les autres sorties, "
          "EXPECTED_SESSION = la valeur donnée en deuxième ligne du rapport, MODE = \"check\" puis \"loop\".")